In [46]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

In [47]:
df = pd.read_excel("2026.02.Fevrier-ExportAccor.xlsx")

In [48]:
df.shape

(3491, 20)

In [49]:
df.head(3)

,NOM BOUTIQUE,OPERATEUR,MACHINE,DATE,HEURE,STATUT,CODE EAN,NOM DU PRODUIT,QUANTITE,PRIX HT,VAT,PRIX TTC,TYPE,GAMME,MARQUE,FOURNISSEUR,ORDER ID (TICKET DE CAISSE),TEMPERATURE,METEO DU JOUR (MOYENNE),METEO DU MOIS (MOYENNE)
0,Novotel Paris Tour Eiffel,DIGITIZME,ACCESSORIES (armoire sèche),2026-02-01,00:11:56,DONE,NaN,ADAPTATEUR UNIVERSEL,1,NaN,19.97,14.0,NON-F&B,SOS,-,-,51847,NaN,NaN,NaN
1,Novotel Paris Tour Eiffel,DIGITIZME,ACCESSORIES (armoire sèche),2026-02-01,00:11:56,DONE,NaN,KIT DENTAIRE COLGATE,1,NaN,20.00,6.0,NON-F&B,SOS,COLGATE,ASTORE,51847,NaN,NaN,NaN
2,Mercure Paris Montmartre Sacré-Cœur,SELFLYSTORE,FRIGO MONTMARTRE (APERO PREMIUM 2),2026-02-01,00:15:28,DONE,3.497915e+12,Chips (90g),1,NaN,5.50,5.0,F&B,FOOD SALEE,-,ASTORE,50790,NaN,NaN,NaN


In [55]:
from turtle import fd


dt_col = "DATE"
tm_col = "HEURE"
dttm_col = "DATETIME"

df[dttm_col] = pd.to_datetime(df[dt_col] + " " + df[tm_col])

df["ANNEE"] = df[dttm_col].dt.year
df["MOIS_D_ANNEE"] = df[dttm_col].dt.month
df["JOUR_DU_MOIS"] = df[dttm_col].dt.day
df["JOUR_DE_SEMAINE"] = df[dttm_col].dt.dayofweek
df["JOUR_D_ANNEE"] = df[dttm_col].dt.dayofyear
df["SEMAINE_D_ANNEE"] = df[dttm_col].dt.isocalendar().week.astype(int)
df["IS_WEEKEND"] = df["JOUR_DE_SEMAINE"].isin([5, 6]).astype(int)
df["HEURE_DU_JOUR"] = df[dttm_col].dt.hour
df.head(3)

df["JOUR_DU_MOIS"].nunique()

for i in range(2, 30):
    df[f"IS_JOUR_DU_MOIS_GEQ_{i}"] = (df["JOUR_DU_MOIS"] >= i).astype(int)

df.head(5)

,NOM BOUTIQUE,OPERATEUR,MACHINE,DATE,HEURE,STATUT,CODE EAN,NOM DU PRODUIT,QUANTITE,PRIX HT,...,IS_JOUR_DU_MOIS_GEQ_20,IS_JOUR_DU_MOIS_GEQ_21,IS_JOUR_DU_MOIS_GEQ_22,IS_JOUR_DU_MOIS_GEQ_23,IS_JOUR_DU_MOIS_GEQ_24,IS_JOUR_DU_MOIS_GEQ_25,IS_JOUR_DU_MOIS_GEQ_26,IS_JOUR_DU_MOIS_GEQ_27,IS_JOUR_DU_MOIS_GEQ_28,IS_JOUR_DU_MOIS_GEQ_29
0,Novotel Paris Tour Eiffel,DIGITIZME,ACCESSORIES (armoire sèche),2026-02-01,00:11:56,DONE,NaN,ADAPTATEUR UNIVERSEL,1,NaN,...,0,0,0,0,0,0,0,0,0,0
1,Novotel Paris Tour Eiffel,DIGITIZME,ACCESSORIES (armoire sèche),2026-02-01,00:11:56,DONE,NaN,KIT DENTAIRE COLGATE,1,NaN,...,0,0,0,0,0,0,0,0,0,0
2,Mercure Paris Montmartre Sacré-Cœur,SELFLYSTORE,FRIGO MONTMARTRE (APERO PREMIUM 2),2026-02-01,00:15:28,DONE,3.497915e+12,Chips (90g),1,NaN,...,0,0,0,0,0,0,0,0,0,0
3,Mercure Paris Montmartre Sacré-Cœur,SELFLYSTORE,FRIGO MONTMARTRE (APERO PREMIUM 2),2026-02-01,00:15:28,DONE,3.292060e+12,Grand vin Mercure,1,NaN,...,0,0,0,0,0,0,0,0,0,0
4,Novotel Paris Tour Eiffel,DIGITIZME,ACCESSORIES (armoire sèche),2026-02-01,00:17:32,DONE,NaN,Jus d'orange Alain Milliat 20cl,1,NaN,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
class Prep():
    def __init__(self, filepath : str):
        self.filepath = filepath

        self._df = None

    @property
    def df(self):
        if(self._df is None):
            self._df = pd.read_excel(self.filepath)
        return self._df
    

    
    


p = Prep("2026.02.Fevrier-ExportAccor.xlsx")
p.df["DATE"]

In [6]:
ORDER_COL = "ORDER ID (TICKET DE CAISSE)"
PRODUCT_COL = "GAMME"
df = (
    df[df["STATUT"].str.upper() == "DONE"][[ORDER_COL, PRODUCT_COL]]
    .dropna()
    .drop_duplicates(subset = [ORDER_COL, PRODUCT_COL])
)


In [6]:
basket = (
    df.assign(value=1)
    .pivot_table(
        index=ORDER_COL,
        columns=PRODUCT_COL,
        values="value",
        aggfunc="max",
        fill_value=0
    )          
    .astype(bool)
)

In [7]:
basket

GAMME,#REF!,ACCESSOIRES,ALCOOL,COSMETIQUE,FOOD SALEE,FOOD SUCREE,JEUX / ENFANTS,PAP,SANS ALCOOL,SOS,SOUVENIRS
ORDER ID (TICKET DE CAISSE),,,,,,,,,,,
50478,False,False,False,False,False,False,False,False,False,True,False
50479,True,False,False,False,False,False,False,False,True,False,False
50480,False,False,False,False,False,False,False,True,True,False,False
50481,False,False,False,False,True,True,False,False,False,False,False
50482,False,False,False,False,False,True,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...
53745,False,False,False,False,True,False,False,False,False,False,False
53746,False,False,False,False,True,False,False,False,True,False,False
53747,False,False,False,False,False,False,False,False,True,False,False


In [8]:
frequent_itemsets = apriori(
    basket,
    min_support=0.01,   # à ajuster selon ton volume
    use_colnames=True
)

In [9]:
frequent_itemsets.shape

(9, 2)

In [10]:
rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1.0
)

In [11]:
rules.shape

(2, 14)

In [1]:
rules

NameError: name 'rules' is not defined

In [13]:
rules[
    (rules["confidence"] >= 0.20) &
    (rules["lift"] >= 1.20)
].sort_values(
    by=["lift", "confidence", "support"],
    ascending=False
)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(FOOD SALEE),(FOOD SUCREE),0.209071,0.211836,0.064159,0.306878,1.448658,1.0,0.019871,1.137122,0.391572,0.179845,0.120587,0.304875
1,(FOOD SUCREE),(FOOD SALEE),0.211836,0.209071,0.064159,0.302872,1.448658,1.0,0.019871,1.134554,0.392946,0.179845,0.118596,0.304875
